
# Tamil Nadu Election 2026 Analysis — TVK vs DMK

This notebook performs:
- Full exploratory data analysis (EDA)
- 10+ visualizations
- TVK vs DMK comparison
- Constituency-level insights
- Basic machine learning analysis

Dataset: `eci_results_tamilnadu_2026.csv`


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10,6)

df = pd.read_csv("eci_results_tamilnadu_2026.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()


## Data Cleaning

In [ ]:

df.columns = [c.strip() for c in df.columns]

numeric_cols = ['EVM Votes', 'Postal Votes', 'Total Votes', '% Votes']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=['Total Votes'])

df.info()


## Identify TVK and DMK Rows

In [ ]:

tvk_df = df[df['Party'].str.contains('TVK|Tamilaga Vettri Kazhagam', case=False, na=False)]
dmk_df = df[df['Party'].str.contains('Dravida Munnetra Kazhagam', case=False, na=False)]

print("TVK rows:", len(tvk_df))
print("DMK rows:", len(dmk_df))


## Visualizations

In [ ]:

# Visualization 1: Top 15 Parties by Total Votes
party_votes = df.groupby('Party')['Total Votes'].sum().sort_values(ascending=False).head(15)

party_votes.plot(kind='bar')
plt.title("Top 15 Parties by Total Votes")
plt.ylabel("Votes")
plt.xticks(rotation=75)
plt.show()

# Visualization 2: TVK vs DMK Total Votes
comparison = pd.DataFrame({
    'Party': ['TVK', 'DMK'],
    'Votes': [tvk_df['Total Votes'].sum(), dmk_df['Total Votes'].sum()]
})

plt.bar(comparison['Party'], comparison['Votes'])
plt.title("TVK vs DMK Total Votes")
plt.ylabel("Votes")
plt.show()

# Visualization 3: Average Vote Percentage
avg_percent = pd.DataFrame({
    'Party': ['TVK', 'DMK'],
    'Avg % Votes': [tvk_df['% Votes'].mean(), dmk_df['% Votes'].mean()]
})

plt.bar(avg_percent['Party'], avg_percent['Avg % Votes'])
plt.title("Average Vote Percentage")
plt.ylabel("Average %")
plt.show()

# Visualization 4: Distribution of Total Votes
df['Total Votes'].hist(bins=50)
plt.title("Distribution of Total Votes")
plt.xlabel("Votes")
plt.ylabel("Frequency")
plt.show()

# Visualization 5: EVM vs Postal Votes
plt.scatter(df['EVM Votes'], df['Postal Votes'])
plt.title("EVM Votes vs Postal Votes")
plt.xlabel("EVM Votes")
plt.ylabel("Postal Votes")
plt.show()

# Visualization 6: Top TVK Constituencies
top_tvk = tvk_df.sort_values('Total Votes', ascending=False).head(10)

plt.barh(top_tvk['Constituency'], top_tvk['Total Votes'])
plt.title("Top 10 TVK Constituencies")
plt.xlabel("Votes")
plt.show()

# Visualization 7: Top DMK Constituencies
top_dmk = dmk_df.sort_values('Total Votes', ascending=False).head(10)

plt.barh(top_dmk['Constituency'], top_dmk['Total Votes'])
plt.title("Top 10 DMK Constituencies")
plt.xlabel("Votes")
plt.show()

# Visualization 8: Vote Percentage Distribution
df['% Votes'].hist(bins=40)
plt.title("Vote Percentage Distribution")
plt.xlabel("% Votes")
plt.ylabel("Frequency")
plt.show()

# Visualization 9: Candidate Count by Party
party_count = df['Party'].value_counts().head(15)

party_count.plot(kind='bar')
plt.title("Top Parties by Candidate Count")
plt.ylabel("Candidates")
plt.xticks(rotation=75)
plt.show()

# Visualization 10: TVK vs DMK Constituency-wise Votes
merged = pd.merge(
    tvk_df[['Constituency', 'Total Votes']],
    dmk_df[['Constituency', 'Total Votes']],
    on='Constituency',
    how='inner',
    suffixes=('_TVK', '_DMK')
)

merged = merged.head(20)

x = np.arange(len(merged))

width = 0.35

fig, ax = plt.subplots(figsize=(14,6))
ax.bar(x - width/2, merged['Total Votes_TVK'], width, label='TVK')
ax.bar(x + width/2, merged['Total Votes_DMK'], width, label='DMK')

ax.set_xticks(x)
ax.set_xticklabels(merged['Constituency'], rotation=90)
ax.legend()
plt.title("TVK vs DMK Constituency Comparison")
plt.show()

# Visualization 11: Correlation Heatmap Data
corr = df[['EVM Votes', 'Postal Votes', 'Total Votes', '% Votes']].corr()

plt.imshow(corr)
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Matrix")
plt.show()


## Machine Learning

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

X = df[['EVM Votes', 'Postal Votes']]
y = df['Total Votes']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("R2 Score:", r2_score(y_test, pred))
print("MAE:", mean_absolute_error(y_test, pred))

plt.scatter(y_test, pred)
plt.xlabel("Actual Votes")
plt.ylabel("Predicted Votes")
plt.title("Actual vs Predicted Total Votes")
plt.show()


## Final Summary

In [ ]:

print("===== SUMMARY =====")

print("\nTotal TVK Votes:", tvk_df['Total Votes'].sum())
print("Total DMK Votes:", dmk_df['Total Votes'].sum())

print("\nAverage TVK Vote %:", round(tvk_df['% Votes'].mean(), 2))
print("Average DMK Vote %:", round(dmk_df['% Votes'].mean(), 2))

print("\nTop TVK Candidate:")
print(tvk_df.sort_values('Total Votes', ascending=False)[['Candidate','Constituency','Total Votes']].head(1))

print("\nTop DMK Candidate:")
print(dmk_df.sort_values('Total Votes', ascending=False)[['Candidate','Constituency','Total Votes']].head(1))
